In [37]:
import sys
import os

# ✅ Convert relative path to absolute, append once
sys.path.append(os.path.abspath("Vintern-1B-v3_5/vintern_local"))

In [35]:
import os
print(os.listdir("Vintern-1B-v3_5/vintern_local"))

['configuration_internvl_chat.py', 'modeling_intern_vit copy.py', 'modeling_intern_vit.py']


In [42]:
# ✅ Now import WITHOUT prefix
# from processing_vinternvl import VinternVLProcessor
# from modeling_intern_vit import InternVLChatModel
from transformers import AutoConfig, BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch

In [43]:
import torch
from transformers import AutoModel, AutoTokenizer

model = AutoModel.from_pretrained(
    "5CD-AI/Vintern-1B-v3_5",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    use_flash_attn=False,
).eval().cuda()

tokenizer = AutoTokenizer.from_pretrained("5CD-AI/Vintern-1B-v3_5", trust_remote_code=True, use_fast=False)


In [45]:
from PIL import Image
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])
    return transform

def load_image(image_file, input_size=448):
    image = Image.open(image_file).convert('RGB')
    transform = build_transform(input_size=input_size)
    image = transform(image)
    return image.unsqueeze(0)  # Add batch dimension


In [ ]:
image_tensor = load_image(r"C:\Users\admin\OneDrive\Desktop\smoking-detection\dataset\images\train\000204_jpg.rf.20e51c2101c8288f663ac7c4914ceec8.jpg").to(torch.bfloat16).cuda()
question = '<image>\nIs the man holding a cigarette or joint in the image?'

generation_config = dict(max_new_tokens=1024, do_sample=False, num_beams=3, repetition_penalty=2.5)

response, history = model.chat(tokenizer, image_tensor, question, generation_config, history=None, return_history=True)
print(f'User: {question}\nAssistant: {response}')



Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


User: <image>
Is the man holding a cigarette or joint in the image?
Assistant: The man is holding a cigarette in the image.
